In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
def transformSilverSip(df, spark):
    # Primeira etapa: tratamento de datas e metadados
    df_tratado = (
        df.withColumn("data_ingestao", to_timestamp(col("data_ingestao"), "yyyy-MM-dd HH:mm:ss"))
          .withColumn("data_modificacao", to_timestamp(col("data_modificacao"), "yyyy-MM-dd HH:mm"))
          .withColumn("camada_origem", lit("bronze"))
          .withColumn("data_tratamento", current_timestamp())
    )

    # Segunda etapa: tratamento de valores nulos e tipos
    sip_tratado = (
        df_tratado.withColumn("PORTE_OPERADORA", when(col("PORTE_OPERADORA") == "0", "NÃO INFORMADO").otherwise(col("PORTE_OPERADORA")))
                  .withColumn("QT_EVENTOS", when(col("QT_EVENTOS") == "nan", 0).otherwise(col("QT_EVENTOS")))
                  .withColumn("QT_BENEF_FORA_CARENCIA", when(col("QT_BENEF_FORA_CARENCIA") == "nan", 0).otherwise(col("QT_BENEF_FORA_CARENCIA")))
                  .withColumn("VL_DESPESA_ASST_LIQ", when(col("VL_DESPESA_ASST_LIQ") == "nan", "0").otherwise(col("VL_DESPESA_ASST_LIQ")))
                  .withColumn("VL_DESPESA_ASST_LIQ", regexp_replace("VL_DESPESA_ASST_LIQ", ",", ".").cast("double"))
    )

    # Terceira etapa: ajustes finais de tipos
    sip_tratato_tipado = (
        sip_tratado.withColumn("QT_EVENTOS", col("QT_EVENTOS").cast(IntegerType()))
                   .withColumn("QT_BENEF_FORA_CARENCIA", col("QT_BENEF_FORA_CARENCIA").cast(IntegerType()))
                   .withColumn("DT_CORTE", to_date(col("DT_CORTE"), "yyyy-MM-dd"))
                   .withColumn("data_tratamento", current_timestamp())
    )

    return sip_tratato_tipado
